In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np

In [2]:
import os
# Read CHROMHMM_PROJECT_DIR and CHROMHMM_GENOME from config.sh (source it before starting the kernel).
_model_dir = os.path.join(os.environ.get("CHROMHMM_PROJECT_DIR", "chromHMM_project"),
                          "chromHMM_out", "model")
bed_path = os.path.join(_model_dir, os.environ.get("CHROMHMM_GENOME", "cho_ont_clean") + "_10_dense.bed")
emission_path = os.path.join(_model_dir, "emissions_10.txt")


In [3]:
# Read states and RGB colors from the BED file
bed_df = pd.read_csv(bed_path, sep="\t", header=None, comment='t')
bed_df = bed_df[[3, 8]]
bed_df.columns = ['StateLabel', 'RGB']
bed_df['StateLabel'] = bed_df['StateLabel'].astype(str)
bed_df['State'] = bed_df['StateLabel'].str.extract(r'(\d+)')
bed_df = bed_df.dropna(subset=['State'])
bed_df['State'] = bed_df['State'].astype(int)


In [4]:
# Obtain the unique state colors
color_df = bed_df.drop_duplicates(subset='State').sort_values('State').reset_index(drop=True)
color_df[['R', 'G', 'B']] = color_df['RGB'].str.split(",", expand=True).astype(int)
color_df['Color'] = color_df.apply(lambda x: (x['R']/255, x['G']/255, x['B']/255), axis=1)

In [5]:
# Add state descriptions and abbreviations
color_df["Description"] = [
    "Repressed PolyComb","Quiescent/Low","Heterochromatin","Strong transcription","Weak transcription","Genic Enhancer","Strong Enhancer",
    "Weak Enhancer","Active Enhancer","Flanking TSS","Bivalent/Poised TSS","Active TSS"
][:len(color_df)]

color_df["Abbreviation"] = ["ReprPC","Quies","Het","Tx","TxWk","EnhG","EnhA","EnhWk","EnhA1","TssFlnk","TssBiv","TssA"][:len(color_df)]

In [6]:
# Read the emission matrix
emission_df = pd.read_csv(emission_path, sep="\t", header=0)
emission_matrix = emission_df.iloc[:, 1:].values
mark_names = emission_df.columns[1:].tolist()
num_states, num_marks = emission_matrix.shape